# LeetCode 1855 — Maximum Distance Between a Pair of Values bằng Python

Link đề: https://leetcode.com/problems/maximum-distance-between-a-pair-of-values/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def maxDistance(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc đúng điều kiện cặp hợp lệ: `i ≤ j` **và** `nums1[i] ≤ nums2[j]`, khoảng cách `j − i`.
2. Khai thác tính **không tăng** (non-increasing) của hai mảng: với `i` cố định, tập `j` thỏa `nums2[j] ≥ nums1[i]` là một **tiền tố** của `nums2` ⇒ tìm kiếm nhị phân.
3. Dùng `bisect` trên mảng **giảm dần** (đảo dấu hoặc đảo mảng) mà không sai chỉ số.
4. Tối ưu thành **hai con trỏ** `O(n + m)`: khi `i` tăng, `j` tốt nhất không giảm.
5. Đối chiếu vét cạn `O(n·m)` trên mảng ngẫu nhiên **không tăng** (seed cố định).

## 2. Đề bài & ví dụ

Cho hai mảng **không tăng** `nums1`, `nums2` (chỉ số từ 0). Cặp `(i, j)` với `0 ≤ i < len(nums1)`, `0 ≤ j < len(nums2)` là **hợp lệ** nếu `i ≤ j` và `nums1[i] ≤ nums2[j]`. **Khoảng cách** của cặp là `j − i`. Trả về khoảng cách lớn nhất của một cặp hợp lệ, hoặc `0` nếu không có cặp nào.

**Ví dụ 1:** `nums1 = [55,30,5,4,2]`, `nums2 = [100,20,10,10,5]` → `2` (cặp `(2,4)`).

**Ví dụ 2:** `nums1 = [2,2,2]`, `nums2 = [10,10,1]` → `1` (cặp `(0,1)`).

**Ví dụ 3:** `nums1 = [30,29,19,5]`, `nums2 = [25,25,25,25,25]` → `2` (cặp `(2,4)`).

**Ràng buộc:** `1 ≤ len(nums1), len(nums2) ≤ 10⁵`; `1 ≤ giá trị ≤ 10⁵`; cả hai mảng không tăng.

In [1]:
import random
import time
import bisect
from typing import List

examples = [([55, 30, 5, 4, 2], [100, 20, 10, 10, 5], 2), ([2, 2, 2], [10, 10, 1], 1), ([30, 29, 19, 5], [25, 25, 25, 25, 25], 2)]
for a, b, exp in examples:
    print(f"nums1={a}, nums2={b} -> kỳ vọng {exp}")

nums1=[55, 30, 5, 4, 2], nums2=[100, 20, 10, 10, 5] -> kỳ vọng 2
nums1=[2, 2, 2], nums2=[10, 10, 1] -> kỳ vọng 1
nums1=[30, 29, 19, 5], nums2=[25, 25, 25, 25, 25] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1 Mảng không tăng và tính chất "tiền tố"

`nums2` không tăng ⇒ với ngưỡng `v`, các vị trí `j` có `nums2[j] ≥ v` tạo thành một **tiền tố** `[0, p)` (khi giá trị rơi xuống dưới `v` thì không bao giờ lên lại).

In [2]:
b = [100, 20, 10, 10, 5]
for v in [5, 10, 11, 101]:
    print(f"v={v}: các j có nums2[j] >= v -> {[j for j in range(len(b)) if b[j] >= v]}")

v=5: các j có nums2[j] >= v -> [0, 1, 2, 3, 4]
v=10: các j có nums2[j] >= v -> [0, 1, 2, 3]
v=11: các j có nums2[j] >= v -> [0, 1]
v=101: các j có nums2[j] >= v -> []


### 3.2 `bisect` trên mảng giảm dần

`bisect` giả định mảng **tăng**. Mẹo: đảo dấu `neg = [-x for x in nums2]` (tăng dần); `nums2[j] ≥ v ⇔ -nums2[j] ≤ -v` ⇒ số phần tử thỏa = `bisect_right(neg, -v)`. Python ≥ 3.10 còn có tham số `key`.

In [3]:
neg = [-x for x in b]
for v in [5, 10, 11, 101]:
    p = bisect.bisect_right(neg, -v)
    print(f"v={v}: tiền tố độ dài {p} -> j lớn nhất = {p - 1}")

v=5: tiền tố độ dài 5 -> j lớn nhất = 4
v=10: tiền tố độ dài 4 -> j lớn nhất = 3
v=11: tiền tố độ dài 2 -> j lớn nhất = 1
v=101: tiền tố độ dài 0 -> j lớn nhất = -1


### 3.3 Hai con trỏ

Hai chỉ số cùng chạy về một hướng, mỗi chỉ số chỉ tiến ⇒ tổng số bước `O(n + m)`.

### 3.4 `max` với giá trị mặc định

Khi có thể không có cặp nào, khởi tạo `best = 0` (đề yêu cầu trả 0).

### 3.5 `bisect` với tham số `key` (Python ≥ 3.10)

Thay vì đảo dấu, có thể cho `bisect` một khóa biến mảng giảm thành "tăng": `bisect_left(nums2, True, key=lambda x: x < v)` — dãy `x < v` có dạng `False … False True … True` trên mảng không tăng, nên chỉ số đầu tiên `True` chính là độ dài tiền tố `nums2[j] ≥ v`.

In [4]:
b = [100, 20, 10, 10, 5]
for v in [5, 10, 11, 101]:
    p = bisect.bisect_left(b, True, key=lambda x: x < v)
    print(f"v={v}: tiền tố nums2 >= v dài {p}")

v=5: tiền tố nums2 >= v dài 5
v=10: tiền tố nums2 >= v dài 4
v=11: tiền tố nums2 >= v dài 2
v=101: tiền tố nums2 >= v dài 0


## 4. Ý tưởng nền tảng / trực giác

Với `i` cố định, muốn `j` **càng lớn càng tốt** sao cho `nums2[j] ≥ nums1[i]` (và `j ≥ i`). Vì `nums2` không tăng, các `j` thỏa là tiền tố ⇒ `j` tốt nhất là **cuối tiền tố**.

Tính tay **Ví dụ 1**:

| i | nums1[i] | j có nums2[j] ≥ nums1[i] | j lớn nhất | j − i (nếu ≥ 0) |
|---|---|---|---|---|
| 0 | 55 | {0} | 0 | 0 |
| 1 | 30 | {0} | 0 | −1 ✗ |
| 2 | 5 | {0,…,4} | 4 | **2** |
| 3 | 4 | {0,…,4} | 4 | 1 |
| 4 | 2 | {0,…,4} | 4 | 0 |

Ngoài ra, khi `i` tăng thì `nums1[i]` **giảm** ⇒ tiền tố thỏa **dài ra** ⇒ `j` tốt nhất **không giảm** — gợi ý hai con trỏ.

In [5]:
a, b, _ = examples[0]
for i, v in enumerate(a):
    js = [j for j in range(len(b)) if b[j] >= v]
    print(i, v, js, "->", max(js) - i if js else None)

0 55 [0] -> 0
1 30 [0] -> -1
2 5 [0, 1, 2, 3, 4] -> 2
3 4 [0, 1, 2, 3, 4] -> 1
4 2 [0, 1, 2, 3, 4] -> 0


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp

In [6]:
def dist_bruteforce(nums1, nums2):
    best = 0
    for i in range(len(nums1)):
        for j in range(i, len(nums2)):
            if nums1[i] <= nums2[j]:
                best = max(best, j - i)
    return best

for a, b, exp in examples:
    got = dist_bruteforce(a, b); print(got, "kỳ vọng", exp); assert got == exp

2 kỳ vọng 2
1 kỳ vọng 1
2 kỳ vọng 2


**Độ phức tạp:** `O(n · m)` = `10¹⁰` ⇒ **TLE**.

**Nút thắt:** không dùng thông tin "không tăng" — với mỗi `i`, ta duyệt cả những `j` chắc chắn không thỏa.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (nhị phân):** với mỗi `i`, `j* = (số phần tử nums2 ≥ nums1[i]) − 1`; nếu `j* ≥ i` thì cập nhật `j* − i`.

**Insight 2 (hai con trỏ):** gọi `J(i) = j*` cho `i`. Vì `nums1[i+1] ≤ nums1[i]`, `J(i+1) ≥ J(i)` ⇒ con trỏ `j` chỉ tiến. Phiên bản gọn: duyệt `i`, `j` cùng lúc:
- **Nếu** `nums1[i] ≤ nums2[j]` **thì** cặp hợp lệ (khi `i ≤ j`), cập nhật đáp án, **tăng `j`** (thử xa hơn).
- **Ngược lại** **tăng `i`** (cần giá trị nhỏ hơn ở `nums1`).

**Invariant:** mọi cặp `(i', j')` với `i' < i` đã được "xét đủ" — đáp án tốt nhất của chúng đã nằm trong `best`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Nhị phân cho mỗi `i` (đảo dấu)

In [7]:
def dist_binary(nums1, nums2):
    neg = [-x for x in nums2]; best = 0
    for i, v in enumerate(nums1):
        j = bisect.bisect_right(neg, -v) - 1
        if j >= i:
            best = max(best, j - i)
    return best

print([dist_binary(a, b) for a, b, _ in examples])

[2, 1, 2]


**Code nộp LeetCode (Cách 2):**

In [8]:
class SolutionBinary:
    def maxDistance(self, nums1: List[int], nums2: List[int]) -> int:
        best = 0
        for i, v in enumerate(nums1):
            lo, hi = i, len(nums2) - 1          # tìm j lớn nhất >= i với nums2[j] >= v
            while lo <= hi:
                mid = (lo + hi) // 2
                if nums2[mid] >= v:
                    best = max(best, mid - i); lo = mid + 1
                else:
                    hi = mid - 1
        return best

print([SolutionBinary().maxDistance(a, b) for a, b, _ in examples])

[2, 1, 2]


**Độ phức tạp Cách 2:** `O(n log m)`, Space `O(1)` (bản tự viết nhị phân) hoặc `O(m)` (bản đảo dấu).

### 7.2 Cách 3 — Nhị phân cho mỗi `j` (tìm `i` nhỏ nhất)

Đối xứng: với `j` cố định, cần `i` **nhỏ nhất** có `nums1[i] ≤ nums2[j]`; vì `nums1` không tăng, các `i` thỏa là **hậu tố** ⇒ nhị phân tìm đầu hậu tố, rồi `i ≤ j`.

In [9]:
def dist_binary_j(nums1, nums2):
    neg1 = [-x for x in nums1]; best = 0
    for j, w in enumerate(nums2):
        i = bisect.bisect_left(neg1, -w)               # i nhỏ nhất có -nums1[i] >= -w  <=>  nums1[i] <= w
        if i < len(nums1) and i <= j:
            best = max(best, j - i)
    return best

print([dist_binary_j(a, b) for a, b, _ in examples])

[2, 1, 2]


**Độ phức tạp Cách 3:** `O(m log n)`.

### 7.3 Cách 4 — Hai con trỏ

In [10]:
def dist_two_pointers(nums1, nums2):
    i = j = best = 0
    while i < len(nums1) and j < len(nums2):
        if nums1[i] <= nums2[j]:
            best = max(best, j - i)
            j += 1
        else:
            i += 1
    return best

print([dist_two_pointers(a, b) for a, b, _ in examples])

[2, 1, 2]


**Code nộp LeetCode (Cách 4):** xem mục 9. **Độ phức tạp:** Time `O(n + m)`, Space `O(1)`.

Lưu ý: khi `i > j`, `best = max(best, j − i)` không ảnh hưởng (âm < 0 ≤ best) nên không cần kiểm tra `i ≤ j` riêng.

### 7.4 Cách 5 — Nhị phân trên **đáp án** `d`

`ok(d)`: tồn tại `i` với `i + d < len(nums2)` và `nums1[i] ≤ nums2[i + d]`. **Đơn điệu:** nếu `(i, i+d)` hợp lệ thì `(i, i+d−1)` cũng hợp lệ (vì `nums2[i+d−1] ≥ nums2[i+d] ≥ nums1[i]`) ⇒ `ok(d) ⇒ ok(d−1)` ⇒ nhị phân tìm `d` lớn nhất. Đây là một ví dụ đẹp của "binary search on answer" khi điều kiện kiểm tra `O(n)`.

In [11]:
class SolutionAnswerBS:
    def maxDistance(self, nums1: List[int], nums2: List[int]) -> int:
        n, m = len(nums1), len(nums2)
        def ok(d):
            return any(nums1[i] <= nums2[i + d] for i in range(min(n, m - d)))
        lo, hi = 0, m - 1
        if not ok(0):
            return 0
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(mid): lo = mid
            else: hi = mid - 1
        return lo

print([SolutionAnswerBS().maxDistance(a, b) for a, b, _ in examples])

[2, 1, 2]


**Độ phức tạp Cách 5:** `O(min(n, m) · log m)`. Khi `ok(0)` sai (không có cặp `(i, i)` hợp lệ) thì cũng không có cặp nào (vì `ok(d) ⇒ ok(0)`) ⇒ trả 0.

## 8. TRACE từng bước

Trace Cách 4 trên **Ví dụ 3** (`nums1 = [30,29,19,5]`, `nums2 = [25]*5`):

| bước | i | j | nums1[i] ≤ nums2[j]? | hành động | best |
|---|---|---|---|---|---|
| 1 | 0 | 0 | 30 ≤ 25 ✗ | i=1 | 0 |
| 2 | 1 | 0 | 29 ≤ 25 ✗ | i=2 | 0 |
| 3 | 2 | 0 | 19 ≤ 25 ✓ | best=max(0,−2)=0, j=1 | 0 |
| 4 | 2 | 1 | ✓ | j − i = −1 không cải thiện, j=2 | 0 |
| 5 | 2 | 2 | ✓ | best=0, j=3 | 0 |
| 6 | 2 | 3 | ✓ | best=1, j=4 | 1 |
| 7 | 2 | 4 | ✓ | best=2, j=5 | **2** |

In [12]:
def trace_two_pointers(nums1, nums2):
    i = j = best = 0; step = 0
    while i < len(nums1) and j < len(nums2):
        step += 1
        ok = nums1[i] <= nums2[j]
        if ok:
            best = max(best, j - i); act = f"hợp lệ, j-i={j - i}, tăng j"
            j += 1
        else:
            act = "vi phạm, tăng i"; i += 1
        print(f"bước {step}: {act:<24} -> i={i}, j={j}, best={best}")
    return best

assert trace_two_pointers([30, 29, 19, 5], [25] * 5) == 2

bước 1: vi phạm, tăng i          -> i=1, j=0, best=0
bước 2: vi phạm, tăng i          -> i=2, j=0, best=0
bước 3: hợp lệ, j-i=-2, tăng j   -> i=2, j=1, best=0
bước 4: hợp lệ, j-i=-1, tăng j   -> i=2, j=2, best=0
bước 5: hợp lệ, j-i=0, tăng j    -> i=2, j=3, best=0
bước 6: hợp lệ, j-i=1, tăng j    -> i=2, j=4, best=1
bước 7: hợp lệ, j-i=2, tăng j    -> i=2, j=5, best=2


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** — hai con trỏ.

In [13]:
class Solution:
    def maxDistance(self, nums1: List[int], nums2: List[int]) -> int:
        i = j = 0
        best = 0
        while i < len(nums1) and j < len(nums2):
            if nums1[i] <= nums2[j]:
                best = max(best, j - i)
                j += 1
            else:
                i += 1
        return best

for a, b, exp in examples:
    got = Solution().maxDistance(a, b); print(got, exp); assert got == exp

2 2
1 1
2 2


## 10. Vì sao thuật toán đúng

Gọi `(i*, j*)` là một cặp tối ưu (nếu có). Ta chứng minh hai con trỏ sẽ đi qua trạng thái `(i, j)` với `i ≤ i*` và `j = j*`, tại đó cặp `(i, j*)` hợp lệ với khoảng cách `≥ j* − i*`.

- `j` chỉ tăng khi cặp hiện tại hợp lệ; `i` chỉ tăng khi `nums1[i] > nums2[j]`.
- **Con trỏ `i` không vượt `i*` trước khi `j` tới `j*`:** `i` chỉ tăng từ giá trị `i*` khi `nums1[i*] > nums2[j]` với một `j < j*`; nhưng `nums2[j] ≥ nums2[j*] ≥ nums1[i*]` (không tăng và `(i*, j*)` hợp lệ) — mâu thuẫn. Vậy khi `j = j*` thì `i ≤ i*`.
- Tại `j = j*`: con trỏ `i` tăng cho tới khi `nums1[i] ≤ nums2[j*]`, và vẫn `≤ i*` (vì `i*` thỏa). Khi đó thuật toán ghi nhận `j* − i ≥ j* − i*`.

Vậy `best ≥` tối ưu; và `best` chỉ ghi nhận cặp hợp lệ (khoảng cách âm không làm tăng `best`) ⇒ `best =` tối ưu. ∎

**Cách 2** đúng trực tiếp: với mỗi `i`, `j` lớn nhất thỏa `nums2[j] ≥ nums1[i]` là cuối tiền tố (Mục 3.1).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums1)`, `m = len(nums2)` (`≤ 10⁵`).

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp | `O(n · m)` | `O(1)` |
| 2. Nhị phân theo `i` | `O(n log m)` | `O(1)` hoặc `O(m)` |
| 3. Nhị phân theo `j` | `O(m log n)` | `O(n)` |
| 4. Hai con trỏ | `O(n + m)` | `O(1)` |
| 5. Nhị phân trên đáp án | `O(min(n,m) log m)` | `O(1)` |

In [14]:
rng = random.Random(1855)
A = sorted((rng.randint(1, 10 ** 5) for _ in range(10 ** 5)), reverse=True)
B = sorted((rng.randint(1, 10 ** 5) for _ in range(10 ** 5)), reverse=True)
for name, f in [("binary", SolutionBinary().maxDistance), ("binary_j", dist_binary_j), ("two_ptr", Solution().maxDistance), ("answerBS", SolutionAnswerBS().maxDistance)]:
    t = time.perf_counter(); r = f(A, B); print(f"{name:<9} -> {r} ({(time.perf_counter()-t)*1000:.0f} ms)")

binary    -> 291 (105 ms)


binary_j  -> 291 (23 ms)
two_ptr   -> 291 (28 ms)
answerBS  -> 291 (101 ms)


## 12. Bộ test

In [15]:
ALL = {"binary": dist_binary, "SolBinary": SolutionBinary().maxDistance, "binary_j": dist_binary_j,
       "two_ptr": dist_two_pointers, "answerBS": SolutionAnswerBS().maxDistance, "submit": Solution().maxDistance}
def run_all(a, b):
    ref = dist_bruteforce(a, b)
    for name, f in ALL.items():
        assert f(a, b) == ref, (name, a, b)
    return ref

for a, b, exp in examples:
    assert run_all(a, b) == exp
edge = [([5], [5], 0), ([5], [4], 0), ([1], [9, 9, 9], 2), ([9, 9, 9], [1], 0), ([3, 2, 1], [3, 2, 1], 0), ([3, 2, 1], [5, 5, 5], 2),
        ([10, 1], [10, 10, 10, 10], 3), ([5, 4], [3, 3, 3], 0)]
for a, b, exp in edge:
    assert run_all(a, b) == exp, (a, b)
rng = random.Random(18550)
for _ in range(1500):
    a = sorted((rng.randint(1, rng.choice([3, 10, 100])) for _ in range(rng.randint(1, 15))), reverse=True)
    b = sorted((rng.randint(1, rng.choice([3, 10, 100])) for _ in range(rng.randint(1, 15))), reverse=True)
    run_all(a, b)
print("Ví dụ, edge case, 1500 test ngẫu nhiên (mảng không tăng): OK")

Ví dụ, edge case, 1500 test ngẫu nhiên (mảng không tăng): OK


### 12.1 Kiểm tra tính chất

1. Đáp án không giảm khi **tăng** một phần tử của `nums2` (vẫn giữ không tăng) hoặc **giảm** một phần tử của `nums1`.
2. Nếu `nums1[0] ≤ nums2[−1]` thì cặp `(0, m − 1)` hợp lệ ⇒ đáp án đạt mức tối đa có thể `m − 1`.

In [16]:
rng = random.Random(41)
for _ in range(500):
    a = sorted((rng.randint(1, 30) for _ in range(rng.randint(1, 12))), reverse=True)
    b = sorted((rng.randint(1, 30) for _ in range(rng.randint(1, 12))), reverse=True)
    base = Solution().maxDistance(a, b)
    b2 = [b[0] + 5] + b[1:]                                  # tăng phần tử đầu (vẫn không tăng)
    assert Solution().maxDistance(a, b2) >= base
    a2 = a[:-1] + [max(1, a[-1] - 3)]                         # giảm phần tử cuối (vẫn không tăng)
    assert Solution().maxDistance(a2, b) >= base
    if a[0] <= b[-1]:
        assert base == len(b) - 1
print("Tính chất đơn điệu và trường hợp cặp xa nhất: OK")

Tính chất đơn điệu và trường hợp cặp xa nhất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên điều kiện `i ≤ j`

In [17]:
def wrong_no_order(nums1, nums2):
    best = 0
    for i in range(len(nums1)):
        for j in range(len(nums2)):
            if nums1[i] <= nums2[j]:
                best = max(best, abs(j - i))               # SAI: cho phép j < i
    return best

a, b, exp = examples[0]
print("sai :", wrong_no_order(a, b), "| đúng:", exp)

sai : 4 | đúng: 2


**Vì sao sai:** cặp `(4, 0)` có `nums1[4] = 2 ≤ 100` nhưng `i > j` — không hợp lệ. **Sửa:** chỉ xét `j ≥ i` (hoặc khoảng cách âm không được tính).

### Lỗi 2 — `bisect` trực tiếp trên mảng giảm dần

In [18]:
def wrong_bisect_desc(nums1, nums2):
    best = 0
    for i, v in enumerate(nums1):
        j = bisect.bisect_right(nums2, v) - 1                # SAI: nums2 không tăng, bisect giả định tăng
        if j >= i: best = max(best, j - i)
    return best

a, b, exp = examples[0]
print("sai :", wrong_bisect_desc(a, b), "| đúng:", exp)

sai : 4 | đúng: 2


**Vì sao sai:** `bisect` chỉ đúng trên mảng **tăng**; trên mảng giảm kết quả vô nghĩa. **Sửa:** đảo dấu, đảo mảng, hoặc tự viết nhị phân.

### Lỗi 3 — Hai con trỏ nhưng tăng `i` khi hợp lệ

In [19]:
def wrong_move_i(nums1, nums2):
    i = j = best = 0
    while i < len(nums1) and j < len(nums2):
        if nums1[i] <= nums2[j]:
            best = max(best, j - i); i += 1                 # SAI: phải thử j xa hơn
        else:
            j += 1
    return best

for a, b, exp in examples:
    print("sai :", wrong_move_i(a, b), "| đúng:", exp)

sai : 0 | đúng: 2
sai : 0 | đúng: 1
sai : 0 | đúng: 2


**Vì sao sai:** khi `(i, j)` hợp lệ, muốn khoảng cách lớn hơn thì phải **tăng `j`**; còn khi vi phạm (giá trị `nums2[j]` quá nhỏ) thì tăng `j` cũng không giúp (nó chỉ nhỏ đi) — phải tăng `i`. Đảo ngược hai nhánh làm bỏ sót cặp tốt. **Sửa:** hợp lệ ⇒ `j += 1`; vi phạm ⇒ `i += 1`.

### Lỗi 4 — "Hai con trỏ" nhưng đặt lại `j = i` mỗi khi tăng `i` (đúng nhưng `O(n·m)`)

In [20]:
def slow_reset_j(nums1, nums2):
    best = 0
    for i in range(len(nums1)):
        j = i                                              # đặt lại: mất lợi thế "j chỉ tiến"
        while j < len(nums2) and nums2[j] >= nums1[i]:
            best = max(best, j - i); j += 1
    return best

A = [1] * 4000; B = [5] * 4000
for f in (slow_reset_j, Solution().maxDistance):
    t = time.perf_counter(); r = f(A, B); print(f"{getattr(f, '__name__', '?'):<12} -> {r} ({(time.perf_counter()-t)*1000:.0f} ms)")
assert slow_reset_j(A, B) == Solution().maxDistance(A, B)

slow_reset_j -> 3999 (571 ms)
maxDistance  -> 3999 (0 ms)


**Vì sao không tốt:** khi mọi cặp hợp lệ, mỗi `i` lại quét gần hết `nums2` ⇒ `O(n·m)`. Tính chất `J(i+1) ≥ J(i)` cho phép **giữ nguyên** `j` khi tăng `i`. **Sửa:** không đặt lại `j` (Cách 4).

### Lỗi 5 — Khởi tạo `best = -1` hoặc trả `None` khi không có cặp

In [21]:
def wrong_init(nums1, nums2):
    best = -1
    for i in range(len(nums1)):
        for j in range(i, len(nums2)):
            if nums1[i] <= nums2[j]: best = max(best, j - i)
    return best

print("sai :", wrong_init([9, 9, 9], [1]), "| đúng:", Solution().maxDistance([9, 9, 9], [1]))

sai : -1 | đúng: 0


**Vì sao sai:** đề yêu cầu trả `0` khi không có cặp hợp lệ. **Sửa:** `best = 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Hai mảng **đã sắp** + tìm cặp tối ưu ⇒ nghĩ ngay **hai con trỏ** hoặc **nhị phân cho mỗi phần tử**.
- Mảng giảm dần ⇒ cẩn thận với `bisect`: đảo dấu / `key=` / tự viết.
- Điều kiện "càng xa càng tốt" ⇒ hợp lệ thì **mở rộng** (tăng `j`), vi phạm thì **thu hẹp** (tăng `i`).
- Luôn kiểm tra ràng buộc thứ tự chỉ số (`i ≤ j`).

## 15. Biến thể của bài

1. **Một mảng duy nhất, `i < j`, `a[i] ≤ a[j]`, mảng bất kỳ**: **962 Maximum Width Ramp** — stack đơn điệu giảm + quét từ phải.
2. **Đếm số cặp hợp lệ**: với mỗi `i`, cộng `max(0, J(i) − i + 1)`.
3. **Không sắp xếp**: sắp chỉ số theo giá trị hoặc dùng stack đơn điệu.
4. **Khoảng cách nhỏ nhất `j − i ≥ 0`**: với mỗi `i`, kiểm tra `j = i` (đơn giản hơn).
5. **Trả về cặp `(i, j)`** đạt khoảng cách lớn nhất, ưu tiên `i` nhỏ (Bài tập 4).

**Minh họa biến thể 2 — đếm số cặp hợp lệ** bằng cùng con trỏ `J(i)` (không giảm theo `i`):

In [22]:
def count_valid_pairs(nums1, nums2):
    total, j = 0, -1
    for i, v in enumerate(nums1):
        if j < i - 1: j = i - 1
        while j + 1 < len(nums2) and nums2[j + 1] >= v:
            j += 1
        total += max(0, j - i + 1)
    return total

rng = random.Random(42)
for _ in range(400):
    a = sorted((rng.randint(1, 9) for _ in range(rng.randint(1, 10))), reverse=True)
    b = sorted((rng.randint(1, 9) for _ in range(rng.randint(1, 10))), reverse=True)
    assert count_valid_pairs(a, b) == sum(1 for i in range(len(a)) for j in range(i, len(b)) if a[i] <= b[j])
print("số cặp hợp lệ ở Ví dụ 1:", count_valid_pairs(*examples[0][:2]), "(đề liệt kê 7 cặp)")

số cặp hợp lệ ở Ví dụ 1: 7 (đề liệt kê 7 cặp)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | Thử hết | `O(nm)` | `O(1)` | Kiểm chứng |
| 2. Nhị phân theo `i` | Cuối tiền tố `nums2 ≥ v` | `O(n log m)` | `O(1)` | Khi chỉ một mảng được sắp |
| 3. Nhị phân theo `j` | Đầu hậu tố `nums1 ≤ w` | `O(m log n)` | `O(n)` | Đối xứng |
| 4. Hai con trỏ | Hợp lệ tăng `j`, vi phạm tăng `i` | `O(n + m)` | `O(1)` | **Bản nộp** |
| 5. Nhị phân đáp án | `ok(d)` đơn điệu | `O(n log m)` | `O(1)` | Luyện binary search on answer |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([5,4], [5,4])`; (b) `([1], [3,2,1])`; (c) `([4,3,2,1], [4,4,4,4])`; (d) `([100], [1])`.

**Bài 2 (tính tay):** Trace hai con trỏ cho Ví dụ 2.

**Bài 3 (điền TODO):** Hoàn thiện `last_ge(nums2, v, lo)` trả `j ≥ lo` lớn nhất có `nums2[j] ≥ v` (hoặc `−1`) bằng nhị phân.

**Bài 4 (tự viết lại):** Viết `best_pair(nums1, nums2)` trả cặp `(i, j)` có khoảng cách lớn nhất (hòa: `i` nhỏ nhất), hoặc `None`.

**Bài 5 (mini challenge):** Viết `max_width_ramp(a)` cho **962**: `max(j − i)` với `i < j`, `a[i] ≤ a[j]`, mảng **bất kỳ**, `O(n)` bằng stack đơn điệu; đối chiếu vét cạn.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def last_ge_todo(nums2, v, lo):
    hi, ans = len(nums2) - 1, -1
    # TODO: nhị phân: nếu nums2[mid] >= v thì ans = mid, lo = mid + 1; ngược lại hi = mid - 1
    return ans

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def best_pair(nums1, nums2):
    # TODO: với mỗi i dùng last_ge; giữ cặp có j - i lớn nhất (cập nhật khi lớn hơn hẳn)
    pass

def max_width_ramp(a):
    # TODO: stack các chỉ số có giá trị giảm dần (từ trái); quét j từ phải, pop khi a[stack[-1]] <= a[j]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: điều kiện `i ≤ j`, `nums1[i] ≤ nums2[j]`. Mức 2: thử `j` lớn nhất. Mức 3: (a) `0`; (b) `2`; (c) `3` (`i = 0`, `4 ≤ 4`, `j = 3`); (d) `0`.

**Bài 2.** Mức 1: `(0,0)`: 2 ≤ 10 ✓, best 0, j=1. Mức 2: `(0,1)`: ✓ best 1, j=2; `(0,2)`: 2 ≤ 1 ✗ ⇒ i=1. Mức 3: `(1,2)` ✗, `(2,2)` ✗ ⇒ hết; đáp án `1`.

**Bài 3.** Mức 1: biên `[lo, len−1]`. Mức 2: điều kiện `nums2[mid] ≥ v`. Mức 3: đối chiếu với quét tuyến tính.

**Bài 4.** Mức 1: `j = last_ge(nums2, nums1[i], i)`. Mức 2: cập nhật khi `j − i > best`. Mức 3: duyệt `i` tăng ⇒ hòa giữ `i` nhỏ.

**Bài 5.** Mức 1: ứng viên `i` tốt chỉ là các "đỉnh giảm dần" từ trái (prefix minimum mới). Mức 2: quét `j` từ phải, khi `a[stack[-1]] ≤ a[j]` thì cập nhật `j − stack.pop()`. Mức 3: mỗi chỉ số push/pop tối đa một lần ⇒ `O(n)`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2, 3
b1 = [Solution().maxDistance(a, b) for a, b in [([5, 4], [5, 4]), ([1], [3, 2, 1]), ([4, 3, 2, 1], [4, 4, 4, 4]), ([100], [1])]]
print("Bài 1:", b1); assert b1 == [0, 2, 3, 0]
print("Bài 2:"); assert trace_two_pointers([2, 2, 2], [10, 10, 1]) == 1
def last_ge_todo(nums2, v, lo):
    hi, ans = len(nums2) - 1, -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums2[mid] >= v: ans, lo = mid, mid + 1
        else: hi = mid - 1
    return ans
rng = random.Random(3)
for _ in range(300):
    b = sorted((rng.randint(1, 9) for _ in range(rng.randint(1, 10))), reverse=True); v = rng.randint(0, 10); lo = rng.randint(0, len(b))
    cands = [j for j in range(lo, len(b)) if b[j] >= v]
    assert last_ge_todo(b, v, lo) == (max(cands) if cands else -1)
print("Bài 3: OK")

Bài 1: [0, 2, 3, 0]
Bài 2:
bước 1: hợp lệ, j-i=0, tăng j    -> i=0, j=1, best=0
bước 2: hợp lệ, j-i=1, tăng j    -> i=0, j=2, best=1
bước 3: vi phạm, tăng i          -> i=1, j=2, best=1
bước 4: vi phạm, tăng i          -> i=2, j=2, best=1
bước 5: vi phạm, tăng i          -> i=3, j=2, best=1
Bài 3: OK


In [26]:
# Bài 4, 5
def best_pair(nums1, nums2):
    best, res = -1, None
    for i, v in enumerate(nums1):
        j = last_ge_todo(nums2, v, i)
        if j != -1 and j - i > best:
            best, res = j - i, (i, j)
    return res

def max_width_ramp(a):
    st = []
    for i, x in enumerate(a):
        if not st or a[st[-1]] > x:
            st.append(i)
    best = 0
    for j in range(len(a) - 1, -1, -1):
        while st and a[st[-1]] <= a[j]:
            best = max(best, j - st.pop())
    return best

rng = random.Random(4)
for _ in range(500):
    a = sorted((rng.randint(1, 9) for _ in range(rng.randint(1, 10))), reverse=True)
    b = sorted((rng.randint(1, 9) for _ in range(rng.randint(1, 10))), reverse=True)
    bp = best_pair(a, b)
    valid = [(i, j) for i in range(len(a)) for j in range(i, len(b)) if a[i] <= b[j]]
    if not valid: assert bp is None
    else:
        m = max(j - i for i, j in valid); assert bp == min(p for p in valid if p[1] - p[0] == m)
    arr = [rng.randint(0, 9) for _ in range(rng.randint(2, 12))]
    assert max_width_ramp(arr) == max([j - i for i in range(len(arr)) for j in range(i + 1, len(arr)) if arr[i] <= arr[j]], default=0)
print("Bài 4, 5: OK |", best_pair(*examples[0][:2]), max_width_ramp([6, 0, 8, 2, 1, 5]))

Bài 4, 5: OK | (2, 4) 4


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template hai con trỏ trên hai mảng không tăng:**

```python
i = j = best = 0
while i < len(nums1) and j < len(nums2):
    if nums1[i] <= nums2[j]:
        best = max(best, j - i); j += 1      # hợp lệ: thử xa hơn
    else:
        i += 1                               # vi phạm: cần nums1 nhỏ hơn
return best
```

**Checklist:**
- [ ] Ràng buộc `i ≤ j`?
- [ ] Hợp lệ ⇒ tăng `j`; vi phạm ⇒ tăng `i`?
- [ ] Không dùng `bisect` thẳng trên mảng giảm dần?
- [ ] Trả `0` khi không có cặp?
- [ ] Đã test một phần tử, không có cặp nào, mọi cặp đều hợp lệ, độ dài lệch nhau?